# Train our own Acehnese → Indonesian model (free Google Colab GPU)
**Before you start:** menu **Runtime → Change runtime type → T4 GPU → Save**. Then **Runtime → Run all**. No API key, nothing to install on your Mac.

What happens (about 30–40 minutes in total):
1. downloads our code and the human-translated pairs (NusaX-MT + FLORES-200, both CC-BY-SA 4.0)
2. builds a leak-checked training set (test sentences can never be trained on)
3. fine-tunes Meta's NLLB-200 (600M, already knows some Acehnese), keeps the best epoch on the *validation* set, stops early when it stops improving
4. scores the **base** and the **fine-tuned** model once on each *test* set (NusaX test, FLORES devtest) and prints a table with examples
5. saves the model to your Google Drive (about 1.2 GB) so the web app can use it

In [ ]:
import os
if not os.path.isdir('/content/aceh_code'):
    !git clone -q -b claude/dataset-product-build-108xu8 https://github.com/Mutaib-yye/aceh_code /content/aceh_code
%cd /content/aceh_code
!git pull -q
!pip install -q -r requirements.txt -U transformers sentencepiece
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo 'NO GPU: Runtime > Change runtime type > T4 GPU'

In [ ]:
# training data: NusaX (+ FLORES-200 if present in data/final/flores) -> data/final/combined
!python -m acehid fetch-nusax
!python -m acehid build-train
import json; m = json.load(open('data/final/combined/manifest.json'))
print('train', m['train']['n'], m['train']['by_source'], '| validation', m['validation']['n'], '| tests', {k: v['n'] for k, v in m['tests'].items()})

**Optional:** add *reviewed* hikayat pairs (a JSON list of `{"ace": ..., "ind": ...}` made by `python -m acehid export`). Machine drafts that nobody checked must not be added. Skip this cell if you have none.

In [ ]:
ADD_REVIEWED_PAIRS = False      # set to True, then run this cell: you will be asked to choose train.json (and test.json)
from google.colab import files
up = files.upload() if ADD_REVIEWED_PAIRS else {}
os.makedirs('data/final/extra', exist_ok=True)
for name, data in up.items():
    open(f'data/final/extra/{name}', 'wb').write(data)
extra = [f'data/final/extra/{n}' for n in up if 'test' not in n]
extra_test = [f'hikayat=data/final/extra/{n}' for n in up if 'test' in n]
if up:
    !python -m acehid build-train --extra {' '.join(extra)} {('--extra-test ' + ' '.join(extra_test)) if extra_test else ''}

In [ ]:
# fine-tune (max 10 epochs, early stopping). Out of GPU memory? add:  --batch-size 4
!python -m acehid train --epochs 10

In [ ]:
from IPython.display import Markdown
Markdown(open('models/ace-id-nllb/RESULTS.md', encoding='utf-8').read())

In [ ]:
# try it: type any Acehnese sentence
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from acehid.normalize import normalize
tok = AutoTokenizer.from_pretrained('models/ace-id-nllb', src_lang='ace_Latn')
model = AutoModelForSeq2SeqLM.from_pretrained('models/ace-id-nllb').to('cuda').eval()
def tr(text):
    x = tok([normalize(text)[0]], return_tensors='pt').to('cuda')
    with torch.no_grad():
        y = model.generate(**x, forced_bos_token_id=tok.convert_tokens_to_ids('ind_Latn'), num_beams=4, max_new_tokens=128)
    return tok.batch_decode(y, skip_special_tokens=True)[0]
for s in ['Lon jak u peukan uroe nyoe.', 'Makanan di sinoe mangat that, tapi harga jih mahai.']:
    print(s, '->', tr(s))

In [ ]:
# save to Google Drive (half precision, ~1.2 GB): Drive > aceh_models > ace-id-nllb
from google.colab import drive
drive.mount('/content/drive')
model.half().save_pretrained('/content/drive/MyDrive/aceh_models/ace-id-nllb'); tok.save_pretrained('/content/drive/MyDrive/aceh_models/ace-id-nllb')
!cp models/ace-id-nllb/RESULTS.md models/ace-id-nllb/training_log.json /content/drive/MyDrive/aceh_models/ace-id-nllb/
print('saved. On the Mac: put that ace-id-nllb folder into aceh_code/models/ and restart ./start.sh; the app then uses YOUR model.')